# MLP: прогноз потребления на сутки вперёд — единый протокол сравнения

Ноутбук использует общие модули проекта (лежат рядом с ним и одинаковы для всех участников):

| Файл | Что делает |
|---|---|
| `preprocessing.py` | загрузка и очистка данных, матрицы «день × час» |
| `split.py` | единое разбиение: скользящее окно на прогнозный период |
| `metrics.py` | единые метрики, режимы «все часы» и «пиковые часы» |
| `mlp_model.py` | только модель MLP (признаки, параметры, обучение) |

## Протокол сравнения
1. **Предобработка** (`preprocessing.py`): нули — это пропуски, заполняются прошлым значением (без заглядывания в будущее). Часы, где факт был нулём, исключаются из метрик.
2. **Разбиение** (`split.py`): прогноз на один год в режиме скользящего окна. Для каждых суток `D` модель **заново обучается** на всех сутках с начала данных до `D−1` включительно. Для 2 января в обучении есть 1 января. Данные суток `D` и позже в обучение не попадают.
3. **Метрики** (`metrics.py`): MAE, RMSE, MAPE в двух режимах — все часы и пиковые часы.
4. **Пиковые часы:** `PEAK_N` самых нагруженных часов каждых суток, определённых **по факту**. Пики нужны только для оценки, при обучении не используются.
5. **Признак `planned_consumption`:** оставлен в общем протоколе, потому что его могут принимать все сравниваемые модели (в Prophet это внешний регрессор `add_regressor`, в XGBoost, LightGBM и RandomForest — обычный признак), а на момент прогноза план известен. Для честного сравнения MLP также считается без плана.
6. **Сравнение с «текущим результатом»:** прежняя постановка — одно обучение на данных до начала прогнозного периода — считается на тех же сутках.

In [1]:
import sys, os
sys.path.insert(0, '../../lib/')                       # модули проекта в lib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

import preprocessing as pp
import split
import metrics
import mlp_model as mm

# ---------------- настройки ----------------
CSV_PATH   = '../../data/clean/75_chelyabinsk.csv'
N_SEEDS    = 3          # число сетей с разной инициализацией (прогнозы усредняются)
WINDOW_DAYS = None      # None - расширяющееся окно; число N - только последние N суток (короткое окно)
N_JOBS     = 1          # -1 = использовать все ядра процессора (ускоряет скользящее окно)
PEAK_N     = metrics.PEAK_N
RECOMPUTE  = False      # True - игнорировать кэш results/ и обучить всё заново

os.makedirs('figures', exist_ok=True)
os.makedirs('results', exist_ok=True)
print('Прогнозный период:', split.FORECAST_START, '...', split.FORECAST_END)

Прогнозный период: 2025-01-01 ... 2025-12-31


# Параметры модели и их обоснование

| Параметр | Значение | Обоснование |
|---|---|---|
| Вход | вчерашние 23 часа + календарь (день недели, день года в sin/cos, флаг выходного) + план на прогнозируемые сутки | всё это известно к моменту прогноза; лаг 7 суток в пробных запусках оказался чуть хуже лага 1 |
| Выход | 23 часа следующих суток одновременно | задача — прогноз всего суточного профиля |
| Слои | (128, 64) | типовая «воронка» для табличной задачи. Перебор 9 архитектур от (16,) до (256, 128) дал MAPE в пределах ~0,02 п.п., то есть размер сети не влияет, и выбрана типовая |
| Активация | ReLU | производная 1 на активной части, поэтому в неглубокой сети градиенты не затухают |
| Оптимизатор | Adam, шаг 1e-3 | устойчивый вариант по умолчанию, сам подбирает масштаб шага для каждого веса |
| Функция ошибки | MSE + L2 (`alpha=1e-3`) | штраф за большие ошибки и умеренная регуляризация против переобучения |
| Ранняя остановка | 10 % обучающих суток под контроль, терпение 30 эпох, максимум 1000 эпох | модель с ~15 тыс. параметров на ~1–1,5 тыс. суток склонна переобучаться |
| Масштабирование | `StandardScaler` на входах и целях, обучается **только на обучающем окне** | значения ~4000 без масштабирования ломают обучение; fit на окне исключает утечку из теста |
| Число сетей | `N_SEEDS` = 3, прогнозы усредняются | снижает случайность от инициализации |

Выбор архитектуры и лага делался на более ранней постановке (валидация — год перед тестом), а не на прогнозном периоде этого протокола.

## 1. Данные

In [2]:
df = pd.read_csv(csv_path, parse_dates=['date'], index_col='date').sort_index()
days, ACTUAL, PLAN, VALID = pp.to_day_matrices(df)

print(f'Данные: {days[0].date()} ... {days[-1].date()}, суток: {len(days)}, часов в сутках: {ACTUAL.shape[1]}')
print(f'Часов с пропущенным фактом (исключены из метрик): {(~VALID).sum()}')
print('Сутки с пропусками:', [d.date().isoformat() for d in days[~VALID.all(axis=1)]])

first = next(split.rolling_origin(days, window_days=WINDOW_DAYS))
last  = list(split.rolling_origin(days, window_days=WINDOW_DAYS))[-1]
print(f'Первые сутки прогноза {days[first[0]].date()}: обучение на {len(first[1])} сутках '
      f'({days[first[1][0]].date()} ... {days[first[1][-1]].date()})')
print(f'Последние сутки прогноза {days[last[0]].date()}: обучение на {len(last[1])} сутках '
      f'({days[last[1][0]].date()} ... {days[last[1][-1]].date()})')

Данные: 2022-01-01 ... 2026-09-23, суток: 1727, часов в сутках: 23
Первые сутки прогноза 2025-01-01: обучение на 1096 сутках (2022-01-01 ... 2024-12-31)
Последние сутки прогноза 2025-12-31: обучение на 1460 сутках (2022-01-01 ... 2025-12-30)


## 2. Обучение: скользящее окно и «текущий результат»

* **Скользящее окно** — новый протокол: каждые сутки прогнозируются моделью, переобученной на всех данных до вчерашнего дня. Это самый долгий шаг: ~365 суток × `N_SEEDS` обучений. Результат сохраняется в `results/`, и повторный запуск ноутбука берёт его из кэша.
* **Одно обучение** — прежняя постановка: модель обучается один раз на данных до начала прогнозного периода. Нужна, чтобы ответить, дало ли скользящее окно улучшение.

In [ ]:
VARIANTS = [('plan',   'MLP + план',     True),
            ('noplan', 'MLP без плана',   False)]

pred_roll, pred_fixed = {}, {}
for key, label, use_plan in VARIANTS:
    X, Y = mm.build_features(days, ACTUAL, PLAN, use_plan=use_plan)
    test_idx, pred_roll[label] = mm.cached_rolling(key, days, X, Y, window_days=WINDOW_DAYS,
                                                   n_seeds=N_SEEDS, n_jobs=N_JOBS, recompute=RECOMPUTE)
    t_fix, pred_fixed[label] = mm.fixed_forecast(days, X, Y, n_seeds=N_SEEDS)
    assert (t_fix == test_idx).all()          # обе постановки - на одних и тех же сутках

test_days = days[test_idx]
y_true, valid = ACTUAL[test_idx], VALID[test_idx]
print(f'Прогнозные сутки: {test_days[0].date()} ... {test_days[-1].date()} ({len(test_days)} шт.)')

Скользящее окно: 365 суток x 3 сетей


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:  3.3min


## 3. Все методы на одних и тех же сутках

In [ ]:
preds = {
    'Вчерашний день':                 ACTUAL[test_idx - 1],
    'Неделю назад':                   ACTUAL[test_idx - 7],
    'План диспетчера':                PLAN[test_idx],
    'MLP без плана (одно обучение)':  pred_fixed['MLP без плана'],
    'MLP + план (одно обучение)':     pred_fixed['MLP + план'],
    'MLP без плана (скользящее окно)': pred_roll['MLP без плана'],
    'MLP + план (скользящее окно)':   pred_roll['MLP + план'],
}
MAIN = 'MLP + план (скользящее окно)'

evals = {k: metrics.evaluate(y_true, p, valid, PEAK_N) for k, p in preds.items()}
modes = list(next(iter(evals.values())).index)
tables = {m: pd.DataFrame({k: e.loc[m] for k, e in evals.items()}).T.round(2) for m in modes}
for m in modes:
    print(f'\n=== {m} ===')
    display(tables[m])
    tables[m].to_csv(f'results/metrics_{"all" if m == "Все часы" else "peak"}.csv')

hit = pd.Series({k: metrics.peak_hit_rate(y_true, p, valid, PEAK_N) for k, p in preds.items()}) * 100
print(f'\nПопадание предсказанного максимума суток в {PEAK_N} фактических пиковых часа, % суток:')
display(hit.round(1).to_frame('попадание, %'))

## 4. Удалось ли улучшить прогноз

In [ ]:
def mape_of(name, mode):
    return tables[mode].loc[name, 'MAPE, %']

pairs = [('MLP + план (скользящее окно)',    'MLP + план (одно обучение)',      'скользящее окно против одного обучения (с планом)'),
         ('MLP без плана (скользящее окно)', 'MLP без плана (одно обучение)',   'скользящее окно против одного обучения (без плана)'),
         ('MLP + план (скользящее окно)',    'План диспетчера',                 'MLP + план (окно) против плана диспетчера'),
         ('MLP без плана (скользящее окно)', 'План диспетчера',                 'MLP без плана (окно) против плана диспетчера'),
         ('MLP + план (скользящее окно)',    'Вчерашний день',                  'MLP + план (окно) против «как вчера»')]

rows = []
for new, old, title in pairs:
    r = {'Сравнение': title}
    for m in modes:
        a, b = mape_of(new, m), mape_of(old, m)
        r[f'{m}: MAPE было → стало, %'] = f'{b:.2f} → {a:.2f}'
        r[f'{m}: изменение ошибки'] = f'{(a - b) / b * 100:+.1f} %'
    rows.append(r)
improvement = pd.DataFrame(rows).set_index('Сравнение')
improvement.to_csv('results/improvement.csv')
improvement

In [ ]:
# Чувствительность вывода к выбору числа пиковых часов N
sens = {}
for n in (1, 3, 5):
    sens[f'N = {n}'] = {k: metrics.evaluate(y_true, preds[k], valid, n).iloc[1]['MAPE, %']
                        for k in ['План диспетчера', 'MLP + план (одно обучение)', MAIN]}
display(pd.DataFrame(sens).round(2).rename_axis('MAPE на пиковых часах, %'))

## 5. Графики

In [ ]:
# Рис. 1. Ошибка по месяцам: все часы и пиковые часы
months = np.arange(1, 13)
plot_models = ['План диспетчера', 'MLP + план (одно обучение)', MAIN]
colors = {'План диспетчера': 'gray', 'MLP + план (одно обучение)': 'tab:blue', MAIN: 'tab:red'}

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=False)
for ax, mode in zip(axes, modes):
    for k in plot_models:
        vals = []
        for mth in months:
            sel = test_days.month == mth
            if not sel.any():
                vals.append(np.nan); continue
            e = metrics.evaluate(y_true[sel], preds[k][sel], valid[sel], PEAK_N)
            vals.append(e.loc[mode, 'MAPE, %'])
        ax.plot(months, vals, 'o-', label=k, color=colors[k])
    ax.set_title(f'MAPE по месяцам — {mode}'); ax.set_xlabel('месяц'); ax.set_ylabel('MAPE, %')
    ax.set_xticks(months); ax.grid(alpha=.3)
axes[0].legend()
plt.tight_layout(); plt.savefig('figures/01_monthly_mape.png', dpi=200); plt.show()

In [ ]:
# Рис. 2. Сравнение методов: все часы и пиковые часы
order = list(preds)
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, mode in zip(axes, modes):
    vals = tables[mode]['MAPE, %'].reindex(order)
    cols = ['#b0b0b0'] * 3 + ['#8fa8c8'] * 2 + ['#e9a3a3', 'tab:red']
    bars = ax.bar(range(len(order)), vals, color=cols)
    ax.bar_label(bars, fmt='%.2f')
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=35, ha='right')
    ax.set_title(f'MAPE, % — {mode}'); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig('figures/02_methods_all_vs_peak.png', dpi=200); plt.show()

In [ ]:
# Рис. 3. Выбранный месяц по часам; пиковые часы отмечены кружками
MONTH = 5
YEAR = test_days[0].year
sel = (test_days.month == MONTH)
d_m, yt, vm = test_days[sel], y_true[sel], valid[sel]
pk = metrics.peak_mask(yt, vm, PEAK_N) & vm

hours_index = pd.DatetimeIndex([d + pd.Timedelta(hours=h) for d in d_m for h in range(pp.HOURS_PER_DAY)])
fact_plot = np.where(vm, yt, np.nan).ravel()          # пропущенный час - разрыв линии

fig, ax = plt.subplots(figsize=(17, 5.5))
ax.plot(hours_index, fact_plot, color='black', lw=1.6, label='Факт')
ax.plot(hours_index, preds[MAIN][sel].ravel(), color='tab:red', lw=1.1, label='MLP + план (скользящее окно)')
ax.plot(hours_index, preds['План диспетчера'][sel].ravel(), color='gray', lw=1, alpha=.8, label='План диспетчера')
ax.scatter(hours_index[pk.ravel()], yt.ravel()[pk.ravel()], s=45, facecolors='none',
           edgecolors='tab:green', linewidths=1.5, label=f'Пиковые часы (топ-{PEAK_N} каждых суток)')
ax.set_title(f'Потребление, {MONTH:02d}.{YEAR}: факт и прогноз на сутки вперёд')
ax.set_ylabel('Потребление'); ax.grid(alpha=.3); ax.legend(ncol=4)
plt.tight_layout(); plt.savefig('figures/03_month_hourly_peaks.png', dpi=200); plt.show()

e_m = {k: metrics.evaluate(yt, p[sel], vm, PEAK_N) for k, p in preds.items()}
month_tbl = pd.concat({m: pd.DataFrame({k: e.loc[m] for k, e in e_m.items()}).T.round(2) for m in modes}, axis=1)
month_tbl.to_csv('results/month_metrics.csv')
month_tbl

In [ ]:
# Рис. 4. Факт против прогноза только на пиковых часах (весь прогнозный период)
pk_all = metrics.peak_mask(y_true, valid, PEAK_N) & valid
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharex=True, sharey=True)
lo, hi = y_true[pk_all].min(), y_true[pk_all].max()
for ax, k in zip(axes, ['План диспетчера', MAIN]):
    ax.scatter(y_true[pk_all], preds[k][pk_all], s=7, alpha=.35)
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
    ax.set_title(f'{k}\nR² на пиковых часах = {r2_score(y_true[pk_all], preds[k][pk_all]):.3f}')
    ax.set_xlabel('Факт'); ax.grid(alpha=.3)
axes[0].set_ylabel('Прогноз')
plt.tight_layout(); plt.savefig('figures/04_peak_scatter.png', dpi=200); plt.show()

In [ ]:
# Рис. 5. Лучшие и худшие сутки выбранного месяца по ошибке в пиковых часах
err_pk = np.where(pk, np.abs(preds[MAIN][sel] - yt) / yt, np.nan)
day_peak_mape = np.nanmean(err_pk, axis=1) * 100
best, worst = int(np.nanargmin(day_peak_mape)), int(np.nanargmax(day_peak_mape))
median = int(np.argsort(day_peak_mape)[len(day_peak_mape) // 2])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharey=True)
for ax, k, name in zip(axes, [best, median, worst], ['Лучшие сутки', 'Типичные сутки', 'Худшие сутки']):
    ax.plot(yt[k], 'k-', lw=2, label='Факт')
    ax.plot(preds[MAIN][sel][k], 'r--', label='MLP + план (окно)')
    ax.plot(preds['План диспетчера'][sel][k], color='gray', ls=':', label='План диспетчера')
    hrs = np.where(pk[k])[0]
    ax.scatter(hrs, yt[k][hrs], s=80, facecolors='none', edgecolors='tab:green', linewidths=2, label='Пиковые часы')
    ax.set_title(f'{name}: {d_m[k]:%d.%m.%Y}\nMAPE в пиковых часах = {day_peak_mape[k]:.2f} %')
    ax.set_xlabel('Час'); ax.grid(alpha=.3)
axes[0].set_ylabel('Потребление'); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig('figures/05_best_worst_days.png', dpi=200); plt.show()

## 6. Сохранение прогнозов в общем формате и итог

In [ ]:
# Прогнозы в «длинном» формате: удобно складывать в общий репозиторий и сравнивать с моделями коллег
long = pd.DataFrame({
    'date': np.repeat(test_days.values, pp.HOURS_PER_DAY),
    'hour': np.tile(np.arange(pp.HOURS_PER_DAY), len(test_days)),
    'actual': y_true.ravel(),
    'actual_valid': valid.ravel(),
    'peak_hour': (metrics.peak_mask(y_true, valid, PEAK_N) & valid).ravel(),
    'pred_mlp_plan': preds['MLP + план (скользящее окно)'].ravel(),
    'pred_mlp_noplan': preds['MLP без плана (скользящее окно)'].ravel(),
})
long.to_csv('results/mlp_predictions.csv', index=False)

m_all, m_pk = modes
print(f'Прогнозный период {test_days[0].date()} ... {test_days[-1].date()}, окно: '
      f'{"расширяющееся" if WINDOW_DAYS is None else str(WINDOW_DAYS) + " суток"}, сетей на день: {N_SEEDS}')
for name in ['План диспетчера', 'MLP + план (одно обучение)', 'MLP + план (скользящее окно)', 'MLP без плана (скользящее окно)']:
    print(f'  {name:35s} MAPE все часы {mape_of(name, m_all):.2f} %, пиковые часы {mape_of(name, m_pk):.2f} %')